# Qwen 5-cohort simple-prompt patching — all score keys (rewrite + logprob + log_delta)

Replays the `female5_patch_male/` sweep with `--score-keys all` so `logprob_delta_scores` is written alongside `rewrite_scores` into a single `aggregate_per_layer.json`. Fresh `--run-id` so the existing rewrite-only run is not touched.

Model: hardcoded `Qwen/Qwen2.5-7B-Instruct` ([simple_patching_without_BHCs.py:35](simple_patching_without_BHCs.py)). No `--model-name` flag exists, so this notebook does **not** produce OLMo log_delta.

**To run end-to-end on Lambda Cloud:** make sure `HF_TOKEN` is exported in the shell before launching Jupyter, then run all cells top-to-bottom. The notebook will (1) print the work shape, (2) verify deps + CUDA, (3) confirm `HF_TOKEN`, (4) run a `--dry-run` and refuse to launch the real run unless exactly 155 units are queued, (5) stream the real run live, (6) assert log_delta is present in the aggregate JSON.

## Cell 1 — Sanity: working dir, branch, files, resolved cohort/prompt lists

In [ ]:
import os, subprocess
from pathlib import Path

# Resolve repo root from this notebook's location.
NB_DIR = Path(os.getcwd()).resolve()
# This notebook lives at <repo>/activation_patching/simple_patching/, so the repo root is two parents up.
REPO_ROOT = NB_DIR.parent.parent if NB_DIR.name == 'simple_patching' else NB_DIR
os.chdir(REPO_ROOT)
print('cwd:', Path.cwd())

print('\n--- git branch ---')
print(subprocess.run(['git', 'branch', '--show-current'], capture_output=True, text=True).stdout.strip())

print('\n--- ls activation_patching/simple_patching/ ---')
for p in sorted(Path('activation_patching/simple_patching').iterdir()):
    kind = 'd' if p.is_dir() else 'f'
    print(f'  {kind} {p.name}')

script = Path('activation_patching/simple_patching/simple_patching_without_BHCs.py')
assert script.exists(), f'patching script missing at {script}'
print(f'\nscript: {script}  ({script.stat().st_size} bytes)')

# Resolved work shape — print explicitly so you can eyeball before launching.
COHORTS    = ['asthma', 'depression', 'multiple_sclerosis', 'rheumatoid_arthritis', 'sarcoidosis']
PROMPT_IDS = list(range(1, 32))  # 1..31 inclusive
OUTPUT_DIR = 'activation_patching/simple_patching'
RUN_ID     = 'female5_patch_male_logdelta'
EXPECTED_UNITS = len(COHORTS) * len(PROMPT_IDS)

print(f'\n--- work shape ---')
print(f'cohorts ({len(COHORTS)}):    {COHORTS}')
print(f'prompt_ids ({len(PROMPT_IDS)}): {PROMPT_IDS}')
print(f'expected units: {len(COHORTS)} cohorts x {len(PROMPT_IDS)} prompts = {EXPECTED_UNITS}')
print(f'run_id:     {RUN_ID}')
print(f'output_dir: {OUTPUT_DIR}/{RUN_ID}')

assert EXPECTED_UNITS == 155, f'expected 155 units (5 x 31), got {EXPECTED_UNITS} — check COHORTS/PROMPT_IDS above'

## Cell 2 — Dependency, CUDA, and HF_TOKEN check

Verifies the runtime can load `nnsight`, `torch`, and `transformers`, that a GPU is visible, and that `HF_TOKEN` is set (Qwen 2.5-7B-Instruct is gated on Hugging Face). If `HF_TOKEN` is missing, set it via the shell before launching Jupyter — or paste it into the noted cell below before running. Do not commit the token.

In [ ]:
import importlib, sys, subprocess, os

needed = ['torch', 'transformers', 'nnsight', 'numpy']
missing = []
for name in needed:
    try:
        importlib.import_module(name)
    except ImportError:
        missing.append(name)

if missing:
    print(f'Installing missing packages: {missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])

import torch, transformers, nnsight
print(f'python      : {sys.version.split()[0]}')
print(f'torch       : {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'nnsight     : {nnsight.__version__}')
print(f'cuda avail  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device 0    : {torch.cuda.get_device_name(0)}')
    free, total = torch.cuda.mem_get_info(0)
    print(f'vram        : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')
else:
    print('WARNING: no CUDA device visible — the sweep will be CPU-only and effectively non-runnable for a 7B model.')

# --- HF_TOKEN check (Qwen 2.5-7B-Instruct is gated) ---
# If HF_TOKEN is not in your env, uncomment the next line and paste your token here.
# os.environ['HF_TOKEN'] = 'hf_xxxxxxxxxxxxxxxxxxxxxxxx'

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    print(f'\nHF_TOKEN     : set (len={len(hf_token)}, starts with {hf_token[:6]}...)')
    # Make it available under both env var names so HF + nnsight see it.
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
else:
    raise RuntimeError(
        'HF_TOKEN is not set. Qwen/Qwen2.5-7B-Instruct is gated and will fail to download. '
        "Set it before launching Jupyter (`export HF_TOKEN=hf_...`) or paste it into this cell "
        "by uncommenting the `os.environ['HF_TOKEN'] = ...` line above, then re-run this cell."
    )

## Cell 3 — Dry run: confirm the script queues exactly 155 units

`--dry-run` validates config and work list without touching the model. This is the 30-second check that the cohort + prompt-id flags were parsed correctly. If this cell asserts on the unit count, **do not proceed to the real run** — fix the flags first.

In [ ]:
import subprocess, sys, re

dry_cmd = [
    sys.executable, '-u',
    'activation_patching/simple_patching/simple_patching_without_BHCs.py',
    '--dry-run',
    '--score-keys',  'all',
    '--cohorts',     ','.join(COHORTS),
    '--prompt-ids',  ','.join(str(i) for i in PROMPT_IDS),
    '--run-id',      RUN_ID,
    '--output-dir',  OUTPUT_DIR,
]
print('DRY-RUN CMD:', ' '.join(dry_cmd), '\n')

result = subprocess.run(dry_cmd, capture_output=True, text=True)
print('--- stdout ---'); print(result.stdout)
if result.stderr:
    print('--- stderr ---'); print(result.stderr)
print(f'\n--- exit code: {result.returncode} ---')
assert result.returncode == 0, 'dry-run failed; check stdout/stderr above before launching the real run'

# Sanity: the script's dry-run output should mention a work list count.
# Look for any integer near 'units' / 'work' wording; otherwise fall back to a generic int scan.
combined = (result.stdout + '\n' + result.stderr)
matches = [int(m) for m in re.findall(r'\b(\d{1,4})\b', combined)]
if 155 in matches:
    print('\nOK: dry-run mentioned the number 155 somewhere in its output — work list size looks right.')
elif 20 in matches and 155 not in matches:
    raise RuntimeError(
        'dry-run output mentions 20 (which is 5 cohorts x 4 default prompt_ids) — the --prompt-ids flag did not take. '
        'Stop here. Verify the prompt-ids list in Cell 1 and re-run.'
    )
else:
    print('\nNOTE: could not verify unit count by string match. Eyeball the stdout above — it should list 155 '
          '(cohort, prompt_id) units across 5 cohorts and 31 prompts. If you see 20 anywhere, abort.')

## Cell 4 — Run the sweep (streams output, tees to log)

Uses `--score-keys all` and the fresh `--run-id` (`female5_patch_male_logdelta`) so the existing rewrite-only bundle at `female5_patch_male/` is not touched. The run is resume-safe via `progress.json` — if the kernel dies, re-running this cell with `--resume` picks up where it left off (any partial pickles already contain all three score matrices, so resume does the right thing here).

Log file: `activation_patching/simple_patching/female5_patch_male_logdelta/run.log`

**Kernel-disconnect risk:** running from Jupyter means a tab close or browser disconnect can kill the kernel and stop the subprocess. For a multi-hour run, prefer launching the equivalent command from a `tmux` session in a terminal (see `RUN_INSTRUCTIONS.md`). The progress.json + resume mechanic protects you, but you'll lose work on the in-flight unit.

In [ ]:
import subprocess, sys, time
from pathlib import Path

RUN_DIR  = Path(OUTPUT_DIR) / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_DIR / 'run.log'

cmd = [
    sys.executable, '-u',
    'activation_patching/simple_patching/simple_patching_without_BHCs.py',
    '--score-keys',  'all',
    '--cohorts',     ','.join(COHORTS),
    '--prompt-ids',  ','.join(str(i) for i in PROMPT_IDS),
    '--run-id',      RUN_ID,
    '--output-dir',  OUTPUT_DIR,
    '--resume',  # safe to leave on; first run has nothing to skip
]
print('CMD:', ' '.join(cmd))
print(f'LOG: {LOG_PATH}\n')

t0 = time.time()
with open(LOG_PATH, 'w') as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        logf.write(line)
        logf.flush()
    proc.wait()
elapsed = time.time() - t0
print(f'\n=== exit code: {proc.returncode}   elapsed: {elapsed/60:.1f} min ===')
assert proc.returncode == 0, 'patching run failed; check log above'

## Cell 5 — Verify log_delta is present in aggregate output

In [ ]:
import json
from pathlib import Path

AGG = Path(OUTPUT_DIR) / RUN_ID / 'aggregate_per_layer.json'
assert AGG.exists(), f'expected aggregate at {AGG}'

data = json.loads(AGG.read_text())
per_layer = data['per_layer']

# Inspect the keys present in a representative row.
sample_keys = sorted(per_layer[0].keys())
print(f'rows in per_layer: {len(per_layer)}')
print(f'keys per row ({len(sample_keys)}):')
for k in sample_keys:
    print(f'  - {k}')

# Hard assertion: log_delta columns must exist.
needed = ['rewrite_scores_mean', 'logprob_scores_mean', 'logprob_delta_scores_mean']
missing_keys = [k for k in needed if k not in per_layer[0]]
assert not missing_keys, f'missing required score keys in aggregate: {missing_keys}'
print(f'\nOK: all three score families present in aggregate_per_layer.json')

# Layer-18 side-by-side comparison.
row_18 = next((r for r in per_layer if r['layer'] == 18), None)
assert row_18 is not None, 'no layer 18 row in aggregate (expected 0..27 for Qwen)'
print(f'\n--- layer 18 (Qwen 28-layer model) ---')
print(f'  rewrite_scores_mean       : {row_18["rewrite_scores_mean"]:.6f}')
print(f'  rewrite_scores_topk_mean  : {row_18["rewrite_scores_topk_mean"]:.6f}')
print(f'  logprob_scores_mean       : {row_18["logprob_scores_mean"]:.6f}')
print(f'  logprob_delta_scores_mean : {row_18["logprob_delta_scores_mean"]:.6f}')
print(f'  logprob_delta_scores_topk : {row_18["logprob_delta_scores_topk_mean"]:.6f}')

## Cell 6 — Notes on re-running / resuming

**The `--resume` flag does NOT retro-add score keys.** It skips any `(cohort, prompt_id)` unit already listed in `progress.json` — and skipped pickles are not re-opened. If you re-run with a `--run-id` whose pickles were written by the original rewrite-only sweep, the resumed run will see all 155 units as complete and produce an `aggregate_per_layer.json` with only `rewrite_scores_*` columns. **You will get the same result as the original run, not a log_delta upgrade.**

To add log_delta to an existing rewrite-only bundle you must:
1. Use a fresh `--run-id` (as this notebook does — `female5_patch_male_logdelta`).
2. Re-execute the full 155-unit model pass.

Resume *within* a `--score-keys all` run is fine: each completed unit's pickle already contains all three score matrices, so partial-progress resume produces a correct aggregate. The caveat applies only to mixing score-key sets across runs that share a run dir.

**OLMo log_delta is not produced by this run.** `MODEL_NAME` is hardcoded to Qwen at line 35 of the script, and there is no `--model-name` CLI flag. Producing the OLMo equivalent is a separate task (edit the constant, fresh `--run-id`).